# APPLE (AAPL) STOCK PRICE PREDICTION
## Next-Day Price Forecasting using Gradient Boosting

This notebook builds a machine learning model to predict AAPL's next-day closing price using Gradient Boosting regression.

## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

In [ ]:
# Load AAPL stock data
df = pd.read_csv('aapl_stock_data.csv')
df['Date'] = pd.to_datetime(df['Date'], utc=True)
df = df.sort_values('Date').reset_index(drop=True)

# Use last 2 years of data
df = df.tail(500).reset_index(drop=True)

print(f"Data Loaded: {len(df)} records")
print(f"Date Range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"\nFirst few rows:")
print(df.head())

## 2. Feature Engineering

In [ ]:
# Lag Features (previous closing prices)
df['Close_Lag1'] = df['Close'].shift(1)
df['Close_Lag2'] = df['Close'].shift(2)
df['Close_Lag3'] = df['Close'].shift(3)
df['Close_Lag5'] = df['Close'].shift(5)

# Moving Averages
df['SMA_5'] = df['Close'].rolling(window=5).mean()
df['SMA_10'] = df['Close'].rolling(window=10).mean()
df['SMA_20'] = df['Close'].rolling(window=20).mean()

# Momentum
df['Momentum_5'] = df['Close'] - df['Close'].shift(5)

# Volatility
df['Returns'] = df['Close'].pct_change()
df['Volatility_5'] = df['Returns'].rolling(window=5).std()

# Volume Indicators
df['Volume_Ratio'] = df['Volume'] / df['Volume'].rolling(window=20).mean()

# Target: Next day's closing price
df['Target'] = df['Close'].shift(-1)

# Remove NaN values
df = df.dropna().reset_index(drop=True)

print(f"Features engineered. Data shape: {df.shape}")
print(f"\nFeatures created:")
features = ['Close_Lag1', 'Close_Lag2', 'Close_Lag3', 'Close_Lag5', 
            'SMA_5', 'SMA_10', 'SMA_20', 'Momentum_5', 'Volatility_5', 'Volume_Ratio']
for feat in features:
    print(f"  ✓ {feat}")

## 3. Data Preparation & Normalization

In [ ]:
# Select features and target
feature_columns = ['Close_Lag1', 'Close_Lag2', 'Close_Lag3', 'Close_Lag5', 
                   'SMA_5', 'SMA_10', 'SMA_20', 'Momentum_5', 'Volatility_5', 'Volume_Ratio']

X = df[feature_columns].values
y = df['Target'].values

# Normalize features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Train-Test Split (80-20, temporal order preserved)
split_point = int(len(X) * 0.80)
X_train, X_test = X_scaled[:split_point], X_scaled[split_point:]
y_train, y_test = y[:split_point], y[split_point:]

print(f"Training set: {len(X_train)} samples (80%)")
print(f"Test set: {len(X_test)} samples (20%)")
print(f"\nTrain period: {df['Date'].iloc[0].date()} to {df['Date'].iloc[split_point-1].date()}")
print(f"Test period:  {df['Date'].iloc[split_point].date()} to {df['Date'].iloc[-1].date()}")

## 4. Model Training

In [ ]:
# Initialize and train Gradient Boosting model
model = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.85,
    min_samples_split=4,
    min_samples_leaf=2,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=42
)

model.fit(X_train, y_train)

print("✓ Gradient Boosting Model Trained")
print(f"\nHyperparameters:")
print(f"  n_estimators: 200")
print(f"  max_depth: 7")
print(f"  learning_rate: 0.05")
print(f"  subsample: 0.85")

## 5. Make Predictions

In [ ]:
# Generate predictions on test set
y_pred = model.predict(X_test)

print(f"Predictions made for {len(y_pred)} test samples")
print(f"\nPrediction Summary:")
print(f"  Actual Min: ${y_test.min():.2f}, Max: ${y_test.max():.2f}, Mean: ${y_test.mean():.2f}")
print(f"  Predicted Min: ${y_pred.min():.2f}, Max: ${y_pred.max():.2f}, Mean: ${y_pred.mean():.2f}")

## 6. Model Evaluation

In [ ]:
# Calculate metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Directional accuracy
direction_actual = np.sign(np.diff(y_test))
direction_pred = np.sign(np.diff(y_pred))
directional_accuracy = np.mean(direction_actual == direction_pred) * 100

print("="*60)
print("MODEL PERFORMANCE METRICS")
print("="*60)
print(f"RMSE (Root Mean Squared Error):  ${rmse:.2f}")
print(f"MAE (Mean Absolute Error):       ${mae:.2f}")
print(f"R² Score:                        {r2:.4f}")
print(f"Directional Accuracy:            {directional_accuracy:.1f}%")
print("="*60)

# Interpretation
avg_price = y_test.mean()
error_percentage = (rmse / avg_price) * 100

print(f"\nINTERPRETATION:")
print(f"  Average test price: ${avg_price:.2f}")
print(f"  Average error: ${rmse:.2f} (~{error_percentage:.2f}% of average price)")
print(f"  Model explains: {r2*100:.2f}% of price movements")
print(f"  Predicts direction correctly: {directional_accuracy:.1f}% of the time")

## 7. Detailed Error Analysis

In [ ]:
# Calculate errors
errors = y_test - y_pred

print("PREDICTION ERROR ANALYSIS:")
print(f"  Mean Error (Bias):        ${errors.mean():.2f}")
print(f"  Std Error:                ${errors.std():.2f}")
print(f"  Min Error:                ${errors.min():.2f}")
print(f"  Max Error:                ${errors.max():.2f}")
print(f"  Median Error:             ${np.median(errors):.2f}")

print(f"\nACCURACY BY ERROR THRESHOLD:")
print(f"  Within $1:   {(np.abs(errors) <= 1).sum():3d}/{len(errors)} ({(np.abs(errors) <= 1).sum()/len(errors)*100:5.1f}%)")
print(f"  Within $2:   {(np.abs(errors) <= 2).sum():3d}/{len(errors)} ({(np.abs(errors) <= 2).sum()/len(errors)*100:5.1f}%)")
print(f"  Within $5:   {(np.abs(errors) <= 5).sum():3d}/{len(errors)} ({(np.abs(errors) <= 5).sum()/len(errors)*100:5.1f}%)")
print(f"  Within 2%:   {(np.abs(errors)/y_test <= 0.02).sum():3d}/{len(errors)} ({(np.abs(errors)/y_test <= 0.02).sum()/len(errors)*100:5.1f}%)")

## 8. Visualization: Actual vs Predicted

In [ ]:
# Plot 1: Time Series
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

test_indices = range(len(y_test))
axes[0,0].plot(test_indices, y_test, label='Actual Price', color='blue', linewidth=2.5, marker='o', markersize=4)
axes[0,0].plot(test_indices, y_pred, label='Predicted Price', color='red', linewidth=2.5, marker='x', markersize=5)
axes[0,0].fill_between(test_indices, y_test, y_pred, alpha=0.15, color='gray')
axes[0,0].set_xlabel('Days in Test Set', fontsize=11, fontweight='bold')
axes[0,0].set_ylabel('Stock Price ($)', fontsize=11, fontweight='bold')
axes[0,0].set_title('AAPL: Actual vs Predicted Next-Day Prices', fontsize=12, fontweight='bold')
axes[0,0].legend(fontsize=10)
axes[0,0].grid(True, alpha=0.3)

# Plot 2: Scatter
axes[0,1].scatter(y_test, y_pred, alpha=0.6, s=60, color='purple', edgecolors='black')
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
axes[0,1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2.5, label='Perfect Prediction')
axes[0,1].set_xlabel('Actual Price ($)', fontsize=11, fontweight='bold')
axes[0,1].set_ylabel('Predicted Price ($)', fontsize=11, fontweight='bold')
axes[0,1].set_title('Prediction Accuracy', fontsize=12, fontweight='bold')
axes[0,1].legend(fontsize=10)
axes[0,1].grid(True, alpha=0.3)

# Plot 3: Error Distribution
axes[1,0].hist(errors, bins=30, color='green', alpha=0.7, edgecolor='black')
axes[1,0].axvline(x=0, color='red', linestyle='--', linewidth=2.5)
axes[1,0].axvline(x=errors.mean(), color='orange', linestyle='--', linewidth=2.5, label=f'Mean: ${errors.mean():.2f}')
axes[1,0].set_xlabel('Prediction Error ($)', fontsize=11, fontweight='bold')
axes[1,0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[1,0].set_title('Distribution of Prediction Errors', fontsize=12, fontweight='bold')
axes[1,0].legend(fontsize=10)
axes[1,0].grid(True, alpha=0.3, axis='y')

# Plot 4: Residuals
axes[1,1].scatter(test_indices, errors, alpha=0.6, s=60, color='darkred', edgecolors='black')
axes[1,1].axhline(y=0, color='green', linestyle='--', linewidth=2)
axes[1,1].fill_between(test_indices, errors, 0, alpha=0.2, color='red')
axes[1,1].set_xlabel('Days in Test Set', fontsize=11, fontweight='bold')
axes[1,1].set_ylabel('Residual Error ($)', fontsize=11, fontweight='bold')
axes[1,1].set_title('Prediction Residuals Over Time', fontsize=12, fontweight='bold')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('stock_prediction_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved as 'stock_prediction_results.png'")

## 9. Feature Importance Analysis

In [ ]:
# Feature importance
importances = model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
sorted_features = [feature_columns[i] for i in sorted_idx]
sorted_importances = importances[sorted_idx]

print("FEATURE IMPORTANCE RANKING:")
print("="*50)
for i, (feat, imp) in enumerate(zip(sorted_features, sorted_importances), 1):
    bar_length = int(imp * 50)
    bar = '█' * bar_length
    print(f"{i:2d}. {feat:15s} {imp:.4f} {bar}")

# Visualization
plt.figure(figsize=(12, 7))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(feature_columns)))
bars = plt.barh(range(len(sorted_features)), sorted_importances, color=colors, edgecolor='black', linewidth=1.5)

plt.yticks(range(len(sorted_features)), sorted_features, fontsize=11, fontweight='bold')
plt.xlabel('Feature Importance Score', fontsize=12, fontweight='bold')
plt.title('Gradient Boosting Model - Feature Importance Ranking', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')

for i, val in enumerate(sorted_importances):
    plt.text(val + 0.003, i, f'{val:.4f}', va='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Feature importance visualization saved as 'feature_importance.png'")

## 10. Export Results

In [ ]:
# Create results dataframe
results_df = pd.DataFrame({
    'Day': range(1, len(y_test) + 1),
    'Actual_Price': y_test.round(2),
    'Predicted_Price': y_pred.round(2),
    'Error_Dollar': errors.round(2),
    'Error_Percentage': (errors / y_test * 100).round(2),
    'Abs_Error': np.abs(errors).round(2)
})

results_df.to_csv('prediction_results.csv', index=False)

print("Sample Results:")
print(results_df.head(10))
print(f"\n✓ Results exported to 'prediction_results.csv'")

## Summary

### Model Performance
- **RMSE**: $7.74 (average prediction error)
- **MAE**: $6.51 (mean absolute error)
- **R² Score**: -0.7105 (explains -71% of variance)
- **Directional Accuracy**: 52.6%

### Key Findings
1. **Feature Importance**: Previous day's closing price (50.66%) is the strongest predictor
2. **Accuracy**: 42.7% of predictions within ±$5 of actual price
3. **Limitations**: Stock prices heavily influenced by external factors not in the model

### Conclusion
While the model achieves directional accuracy slightly better than random guessing, it highlights the difficulty of predicting financial markets using only historical price data. Successful trading systems require integration of multiple data sources and sophisticated risk management.